In [3]:
import os
import glob
import json
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Ready.")

Ready.


In [6]:
ROOT = "/kaggle/input"

all_csv = []

for root, dirs, files in os.walk(ROOT):
    for file in files:
        if file.lower().endswith(".csv"):
            all_csv.append(
                os.path.join(root, file)
            )

print("CSV files found:", len(all_csv))

for f in all_csv[:30]:
    print(f)

CSV files found: 11819
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45/ata_read.csv
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45/ata_write.csv
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45/mem_exec.csv
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45/mem_readwrite.csv
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45/mem_read.csv
/kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMA

In [7]:
trace_files = {
    "ata_read": [],
    "ata_write": [],
    "mem_read": [],
    "mem_write": [],
    "mem_readwrite": [],
    "mem_exec": []
}

for path in all_csv:

    name = os.path.basename(path).lower()

    for trace_type in trace_files:

        if name == trace_type + ".csv":
            trace_files[trace_type].append(path)

for key, files in trace_files.items():

    print(
        f"{key:15s}: {len(files)}"
    )

ata_read       : 1969
ata_write      : 1970
mem_read       : 1970
mem_write      : 1970
mem_readwrite  : 1970
mem_exec       : 1970


In [8]:
for trace_type, files in trace_files.items():

    if files:

        print(
            "\n==========",
            trace_type,
            "=========="
        )

        df = pd.read_csv(
            files[0],
            header=None
        )

        print(
            "Shape:",
            df.shape
        )

        display(
            df.head()
        )


========== ata_read ==========
Shape: (1523441, 6)


,0,1,2,3,4,5
0,1709002891,651048440,1372256,4096,-1,0
1,1709002891,651048440,1372264,4096,-1,0
2,1709002891,651049513,1372272,4096,-1,0
3,1709002891,651049513,1372280,4096,-1,0
4,1709002891,651050587,1372288,4096,-1,0



========== ata_write ==========
Shape: (1203962, 6)


,0,1,2,3,4,5
0,1709002892,453478,50318280,4096,0.368830,0
1,1709002892,993449,50369384,4096,0.251421,0
2,1709002892,1535566,50370120,4096,0.402355,0
3,1709002892,1811456,50415568,4096,0.261254,0
4,1709002892,2377190,50415696,4096,0.251293,0



========== mem_read ==========
Shape: (581685, 6)


...

,0,1,2,3,4,5
0,1709002890,894292018,7574277128,4096,0.772902,2
1,1709002890,897849597,4276093104,4096,0.170929,2
2,1709002890,898225322,19340146792,4096,0.844234,2
3,1709002890,898529122,7539017024,4096,0.784924,2
4,1709002890,902388355,18722856,4096,0.854214,2



========== mem_readwrite ==========
Shape: (711, 6)


,0,1,2,3,4,5
0,1709002890,903254669,8169501772,4096,0.759721,2
1,1709002890,905141882,8104414640,4096,0.845544,2
2,1709002890,905606708,4305522692,4096,0.994323,2
3,1709002890,919507459,7691000156,4096,0.788934,2
4,1709002890,921075843,8342102020,4096,0.739084,2



========== mem_exec ==========
Shape: (219, 6)


,0,1,2,3,4,5
0,1709002890,894308120,41766416,0,-1,2
1,1709002890,894367163,42007460,0,-1,2
2,1709002890,894376824,50889824,0,-1,2
3,1709002890,898296173,19569410400,0,-1,2
4,1709002890,898316569,18259130848,0,-1,2


In [10]:
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------
# Build sample directory map
# ------------------------------------------------------------

sample_dirs = defaultdict(dict)

for trace_type, files in trace_files.items():

    for path in files:

        p = Path(path)

        # The directory containing the six CSV files
        sample_dir = p.parent

        sample_dirs[str(sample_dir)][trace_type] = str(p)


print("Total sample directories:", len(sample_dirs))

print("\nExample:")
for sample_dir, traces in list(sample_dirs.items())[:3]:

    print("\n", sample_dir)

    for trace_type, path in traces.items():
        print("  ", trace_type, "->", Path(path).name)

Total sample directories: 1970

Example:

 /kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-16-45
   ata_read -> ata_read.csv
   ata_write -> ata_write.csv
   mem_read -> mem_read.csv
   mem_write -> mem_write.csv
   mem_readwrite -> mem_readwrite.csv
   mem_exec -> mem_exec.csv

 /kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-33-31
   ata_read -> ata_read.csv
   ata_write -> ata_write.csv
   mem_read -> mem_read.csv
   mem_write -> mem_write.csv
   mem_readwrite -> mem_readwrite.csv
   mem_exec -> mem_exec.csv

 /kaggle/input/datasets/hiranomanabu/ransmap-2024-ransomware-behavioral-features/RanSMAP/dataset/variants/i3-gen12/ddr4-2133-16g/Conti_03/Conti_03-20240227_21-08-44
   ata_read -> ata_read.csv
   ata_write -> ata_write.csv
   mem_read -> mem_read.csv
   mem_writ

In [12]:
# ============================================================
# CELL 6 — FINAL LABELING
# ============================================================

RANSOMWARE_APPS = {
    "LockBit",
    "Darkside",
    "Ryuk",
    "AESCrypt",
    "REvil",
    "WannaCry",
    "Conti"
}

BENIGN_APPS = {
    "Idle",
    "Firefox",
    "SDelete",
    "Zip",
    "Office"
}


def get_dataset_group(path):

    parts = Path(path).parts

    for group in [
        "original",
        "variants",
        "mix",
        "extra"
    ]:

        if group in parts:
            return group

    return "unknown"


def get_application(path):

    p = Path(path)

    # Example:
    # .../original/.../LockBit/LockBit-2024...
    # application = LockBit

    return p.parent.name


def get_label(path):

    group = get_dataset_group(path)
    application = get_application(path)

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    if group == "original":

        if application in RANSOMWARE_APPS:
            return 1

        if application in BENIGN_APPS:
            return 0

        return None


    # --------------------------------------------------------
    # VARIANTS
    # --------------------------------------------------------

    if group == "variants":

        # All variants are Conti ransomware variants
        return 1


    # --------------------------------------------------------
    # MIX
    # --------------------------------------------------------

    if group == "mix":

        # Every mix contains a ransomware application
        return 1


    # --------------------------------------------------------
    # EXTRA
    # --------------------------------------------------------

    if group == "extra":

        # Label it correctly, but DO NOT use it for training.
        if application in RANSOMWARE_APPS:
            return 1

        if application in BENIGN_APPS:
            return 0

        return None


    return None


# ============================================================
# BUILD METADATA
# ============================================================

sample_metadata = []

for sample_dir in sample_dirs:

    group = get_dataset_group(
        sample_dir
    )

    application = get_application(
        sample_dir
    )

    label = get_label(
        sample_dir
    )

    sample_metadata.append({

        "sample_dir":
            sample_dir,

        "group":
            group,

        "application":
            application,

        "label":
            label
    })


metadata_df = pd.DataFrame(
    sample_metadata
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("========== GROUP COUNTS ==========")

print(
    metadata_df[
        "group"
    ].value_counts()
)


print(
    "\n========== LABEL COUNTS =========="
)

print(
    metadata_df[
        "label"
    ].value_counts(
        dropna=False
    )
)


print(
    "\n========== GROUP + LABEL =========="
)

print(
    pd.crosstab(
        metadata_df["group"],
        metadata_df["label"],
        dropna=False
    )
)


print(
    "\n========== APPLICATION + LABEL =========="
)

print(
    metadata_df[
        [
            "group",
            "application",
            "label"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "group",
            "application"
        ]
    )
    .to_string(
        index=False
    )
)

========== GROUP COUNTS ==========
group
original    1440
extra        360
mix          100
variants      70
Name: count, dtype: int64

========== LABEL COUNTS ==========
label
1    1220
0     750
Name: count, dtype: int64

========== GROUP + LABEL ==========
label       0    1
group             
extra     150  210
mix         0  100
original  600  840
variants    0   70

========== APPLICATION + LABEL ==========
   group          application  label
   extra             AESCrypt      1
   extra                Conti      1
   extra             Darkside      1
   extra              Firefox      0
   extra                 Idle      0
   extra              LockBit      1
   extra               Office      0
   extra                REvil      1
   extra                 Ryuk      1
   extra              SDelete      0
   extra             WannaCry      1
   extra                  Zip      0
     mix       AESCrypt_Conti      1
     mix       AESCrypt_REvil      1
     mix        Firefox_Cont

In [20]:
# ============================================================
# CELL 7 — EFFICIENT TRACE FEATURE EXTRACTION
# ============================================================

def process_trace(
    path,
    trace_type,
    chunksize=250_000
):
    """
    Process one RanSMAP trace CSV in chunks.

    This avoids loading million-row CSV files
    completely into RAM.
    """

    if path is None:
        return None

    operation_count = 0
    total_size = 0.0
    size_sum_sq = 0.0

    min_time = None
    max_time = None

    unique_addresses = set()

    min_address = None
    max_address = None

    entropy_sum = 0.0
    entropy_sum_sq = 0.0
    entropy_count = 0
    entropy_max = 0.0

    try:

        for chunk in pd.read_csv(
            path,
            header=None,
            chunksize=chunksize
        ):

            if chunk.empty:
                continue

            # ------------------------------------------------
            # TIME
            # ------------------------------------------------

            times = pd.to_numeric(
                chunk.iloc[:, 0],
                errors="coerce"
            ).dropna()

            if len(times) > 0:

                current_min = times.min()
                current_max = times.max()

                if (
                    min_time is None
                    or current_min < min_time
                ):
                    min_time = current_min

                if (
                    max_time is None
                    or current_max > max_time
                ):
                    max_time = current_max


            # ------------------------------------------------
            # SIZE
            # ------------------------------------------------

            sizes = pd.to_numeric(
                chunk.iloc[:, 3],
                errors="coerce"
            ).dropna()

            if len(sizes) > 0:

                operation_count += len(sizes)

                total_size += sizes.sum()

                size_sum_sq += (
                    sizes.astype(float) ** 2
                ).sum()


            # ------------------------------------------------
            # ADDRESS
            #
            # ATA -> LBA
            # MEM -> GPA
            # ------------------------------------------------

            addresses = pd.to_numeric(
                chunk.iloc[:, 2],
                errors="coerce"
            ).dropna()

            if len(addresses) > 0:

                unique_addresses.update(
                    addresses.astype(
                        np.int64
                    ).tolist()
                )

                current_min = addresses.min()
                current_max = addresses.max()

                if (
                    min_address is None
                    or current_min < min_address
                ):
                    min_address = current_min

                if (
                    max_address is None
                    or current_max > max_address
                ):
                    max_address = current_max


            # ------------------------------------------------
            # ENTROPY
            # ------------------------------------------------

            if trace_type in [
                "ata_write",
                "mem_write",
                "mem_readwrite"
            ]:

                entropy = pd.to_numeric(
                    chunk.iloc[:, 4],
                    errors="coerce"
                ).dropna()

                if len(entropy) > 0:

                    entropy_sum += entropy.sum()

                    entropy_sum_sq += (
                        entropy.astype(float) ** 2
                    ).sum()

                    entropy_count += len(entropy)

                    entropy_max = max(
                        entropy_max,
                        entropy.max()
                    )


    except Exception as e:

        print(
            f"ERROR processing {path}: {e}"
        )

        return None


    if operation_count == 0:
        return None


    # --------------------------------------------------------
    # DURATION
    # --------------------------------------------------------

    if (
        min_time is not None
        and max_time is not None
    ):

        duration = max(
            float(max_time - min_time),
            1e-6
        )

    else:

        duration = 1e-6


    # --------------------------------------------------------
    # SIZE STATISTICS
    # --------------------------------------------------------

    avg_size = (
        total_size /
        operation_count
    )


    variance = (
        size_sum_sq /
        operation_count
    ) - (
        avg_size ** 2
    )

    variance = max(
        variance,
        0.0
    )


    std_size = np.sqrt(
        variance
    )


    # --------------------------------------------------------
    # ENTROPY STATISTICS
    # --------------------------------------------------------

    if entropy_count > 0:

        entropy_mean = (
            entropy_sum /
            entropy_count
        )

    else:

        entropy_mean = 0.0


    if entropy_count > 1:

        entropy_variance = (
            entropy_sum_sq /
            entropy_count
        ) - (
            entropy_mean ** 2
        )

        entropy_variance = max(
            entropy_variance,
            0.0
        )

        entropy_std = np.sqrt(
            entropy_variance
        )

    else:

        entropy_std = 0.0


    # --------------------------------------------------------
    # RETURN FEATURES
    # --------------------------------------------------------

    return {

        "operation_count":
            operation_count,

        "total_bytes":
            total_size,

        "avg_operation_size":
            avg_size,

        "std_operation_size":
            std_size,

        "duration":
            duration,

        "operation_velocity":
            operation_count / duration,

        "unique_address_count":
            len(unique_addresses),

        "address_range":
            (
                max_address - min_address
                if (
                    min_address is not None
                    and max_address is not None
                )
                else 0
            ),

        "entropy_mean":
            entropy_mean,

        "entropy_std":
            entropy_std,

        "entropy_max":
            entropy_max
    }

In [21]:
# ============================================================
# CELL 8 — BUILD SAMPLE-LEVEL BEHAVIORAL PROFILE
# ============================================================

def build_sample_features(
    sample_dir,
    traces
):

    result = {
        "sample_dir": sample_dir
    }

    trace_results = {}


    # --------------------------------------------------------
    # Process all six telemetry sources
    # --------------------------------------------------------

    for trace_type in trace_files.keys():

        path = traces.get(
            trace_type
        )

        features = process_trace(
            path,
            trace_type
        )


        # If trace is empty/unavailable,
        # use safe zero values.

        if features is None:

            features = {

                "operation_count": 0,
                "total_bytes": 0,
                "avg_operation_size": 0,
                "std_operation_size": 0,
                "duration": 0,
                "operation_velocity": 0,
                "unique_address_count": 0,
                "address_range": 0,
                "entropy_mean": 0,
                "entropy_std": 0,
                "entropy_max": 0

            }


        trace_results[
            trace_type
        ] = features


    # --------------------------------------------------------
    # Overall activity
    # --------------------------------------------------------

    all_features = list(
        trace_results.values()
    )


    result[
        "total_operations"
    ] = sum(
        x["operation_count"]
        for x in all_features
    )


    result[
        "total_bytes"
    ] = sum(
        x["total_bytes"]
        for x in all_features
    )


    result[
        "total_operation_velocity"
    ] = sum(
        x["operation_velocity"]
        for x in all_features
    )


    result[
        "total_unique_addresses"
    ] = sum(
        x["unique_address_count"]
        for x in all_features
    )


    # --------------------------------------------------------
    # Preserve individual telemetry sources
    # --------------------------------------------------------

    for trace_type, features in trace_results.items():

        prefix = trace_type


        result[
            f"{prefix}_operations"
        ] = features[
            "operation_count"
        ]


        result[
            f"{prefix}_bytes"
        ] = features[
            "total_bytes"
        ]


        result[
            f"{prefix}_velocity"
        ] = features[
            "operation_velocity"
        ]


        result[
            f"{prefix}_unique_addresses"
        ] = features[
            "unique_address_count"
        ]


        result[
            f"{prefix}_address_range"
        ] = features[
            "address_range"
        ]


        result[
            f"{prefix}_entropy_mean"
        ] = features[
            "entropy_mean"
        ]


        result[
            f"{prefix}_entropy_std"
        ] = features[
            "entropy_std"
        ]


        result[
            f"{prefix}_entropy_max"
        ] = features[
            "entropy_max"
        ]


    return result

In [23]:
# ============================================================
# CELL 9 — EXTRACT TRAINING DATA
# ============================================================

feature_records = []

total_samples = len(
    sample_dirs
)


for index, (
    sample_dir,
    traces
) in enumerate(
    sample_dirs.items()
):

    if index % 25 == 0:

        print(
            f"Processing "
            f"{index}/{total_samples}"
        )


    group = get_dataset_group(
        sample_dir
    )

    label = get_label(
        sample_dir
    )


    # --------------------------------------------------------
    # Skip unknown labels
    # --------------------------------------------------------

    if label is None:
        continue


    # --------------------------------------------------------
    # IMPORTANT:
    # EXTRA IS NEVER USED FOR TRAINING
    # --------------------------------------------------------

    if group == "extra":
        continue


    # --------------------------------------------------------
    # Extract behavioral features
    # --------------------------------------------------------

    features = build_sample_features(
        sample_dir,
        traces
    )


    features["group"] = group

    features["label"] = label


    feature_records.append(
        features
    )


# ------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------

behavior_df = pd.DataFrame(
    feature_records
)


print("\n================================")
print("EXTRACTION COMPLETE")
print("================================")

print(
    "Training samples:",
    len(behavior_df)
)


print("\nGroups:")

print(
    behavior_df[
        "group"
    ].value_counts()
)


print("\nLabels:")

print(
    behavior_df[
        "label"
    ].value_counts()
)

Processing 0/1970
Processing 25/1970
Processing 50/1970
Processing 75/1970
Processing 100/1970
Processing 125/1970
Processing 150/1970
Processing 175/1970
Processing 200/1970
Processing 225/1970
Processing 250/1970
Processing 275/1970
Processing 300/1970
Processing 325/1970
Processing 350/1970
Processing 375/1970
Processing 400/1970
Processing 425/1970
Processing 450/1970
Processing 475/1970
Processing 500/1970
Processing 525/1970
Processing 550/1970
Processing 575/1970
Processing 600/1970
Processing 625/1970
Processing 650/1970
Processing 675/1970
Processing 700/1970
Processing 725/1970
Processing 750/1970
Processing 775/1970
Processing 800/1970
Processing 825/1970
Processing 850/1970
Processing 875/1970
Processing 900/1970
Processing 925/1970
Processing 950/1970
Processing 975/1970
Processing 1000/1970
Processing 1025/1970
Processing 1050/1970
Processing 1075/1970
Processing 1100/1970
Processing 1125/1970
Processing 1150/1970
Processing 1175/1970
Processing 1200/1970
Processing 1225/

In [24]:
# ============================================================
# CELL 10 — CLEAN AND SAVE
# ============================================================

behavior_df = behavior_df.replace(
    [np.inf, -np.inf],
    np.nan
)


behavior_df = behavior_df.fillna(
    0
)


TRAINING_DATA_PATH = (
    "/kaggle/working/"
    "ransmap_behavior_features.csv"
)


behavior_df.to_csv(
    TRAINING_DATA_PATH,
    index=False
)


print(
    "Saved:",
    TRAINING_DATA_PATH
)


print(
    "Shape:",
    behavior_df.shape
)


display(
    behavior_df.head()
)

Saved: /kaggle/working/ransmap_behavior_features.csv
Shape: (1610, 55)


,sample_dir,total_operations,total_bytes,total_operation_velocity,total_unique_addresses,ata_read_operations,ata_read_bytes,ata_read_velocity,ata_read_unique_addresses,ata_read_address_range,...,mem_exec_operations,mem_exec_bytes,mem_exec_velocity,mem_exec_unique_addresses,mem_exec_address_range,mem_exec_entropy_mean,mem_exec_entropy_std,mem_exec_entropy_max,group,label
0,/kaggle/input/datasets/hiranomanabu/ransmap-20...,3933782,1.113056e+10,20310.056188,2620472,1523441,6.208927e+09,7852.788660,1520596,169053912,...,219,0.0,1.177419,218,19527643984,0.0,0.0,0.0,variants,1
1,/kaggle/input/datasets/hiranomanabu/ransmap-20...,3787129,1.076001e+10,19964.773265,2527716,1463665,5.964715e+09,7703.500000,1460445,169053912,...,206,0.0,1.138122,204,19379544372,0.0,0.0,0.0,variants,1
2,/kaggle/input/datasets/hiranomanabu/ransmap-20...,3407976,9.573259e+09,18254.379450,2266261,1309777,5.336814e+09,7004.155080,1306572,169053912,...,197,0.0,1.172619,196,19381718280,0.0,0.0,0.0,variants,1
3,/kaggle/input/datasets/hiranomanabu/ransmap-20...,4063697,1.150865e+10,21776.571217,2696699,1569736,6.397476e+09,8439.440860,1566956,169053912,...,232,0.0,1.681159,232,19527971616,0.0,0.0,0.0,variants,1
4,/kaggle/input/datasets/hiranomanabu/ransmap-20...,3804534,1.079961e+10,20204.098165,2535644,1465421,5.972164e+09,7794.792553,1462300,169053912,...,218,0.0,1.159574,218,19561201556,0.0,0.0,0.0,variants,1


In [26]:
# ============================================================
# CELL 11 — FINAL FEATURE SCHEMA
# ============================================================

FINAL_FEATURES = [

    # Overall activity
    "total_operations",
    "total_bytes",
    "total_operation_velocity",
    "total_unique_addresses",

    # ATA read/write
    "ata_read_operations",
    "ata_write_operations",

    "ata_read_bytes",
    "ata_write_bytes",

    "ata_read_velocity",
    "ata_write_velocity",

    # Memory activity
    "mem_read_operations",
    "mem_write_operations",
    "mem_readwrite_operations",
    "mem_exec_operations",

    "mem_read_velocity",
    "mem_write_velocity",
    "mem_readwrite_velocity",
    "mem_exec_velocity",

    # ATA write entropy
    "ata_write_entropy_mean",
    "ata_write_entropy_std",
    "ata_write_entropy_max",

    # Memory write entropy
    "mem_write_entropy_mean",
    "mem_write_entropy_std",
    "mem_write_entropy_max",

    # Memory read/write entropy
    "mem_readwrite_entropy_mean",
    "mem_readwrite_entropy_std",
    "mem_readwrite_entropy_max"
]


X = behavior_df[
    FINAL_FEATURES
]


y = behavior_df[
    "label"
].astype(int)


print(
    "Number of features:",
    len(FINAL_FEATURES)
)

print(
    "Number of samples:",
    len(X)
)

print(
    "\nFeature names:"
)

for i, feature in enumerate(
    FINAL_FEATURES,
    start=1
):

    print(
        i,
        feature
    )

Number of features: 27
Number of samples: 1610

Feature names:
1 total_operations
2 total_bytes
3 total_operation_velocity
4 total_unique_addresses
5 ata_read_operations
6 ata_write_operations
7 ata_read_bytes
8 ata_write_bytes
9 ata_read_velocity
10 ata_write_velocity
11 mem_read_operations
12 mem_write_operations
13 mem_readwrite_operations
14 mem_exec_operations
15 mem_read_velocity
16 mem_write_velocity
17 mem_readwrite_velocity
18 mem_exec_velocity
19 ata_write_entropy_mean
20 ata_write_entropy_std
21 ata_write_entropy_max
22 mem_write_entropy_mean
23 mem_write_entropy_std
24 mem_write_entropy_max
25 mem_readwrite_entropy_mean
26 mem_readwrite_entropy_std
27 mem_readwrite_entropy_max


In [29]:
# ============================================================
# CELL 12 — TRAIN / VALIDATION / TEST
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(

    X,
    y,

    test_size=0.30,

    random_state=42,

    stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(

    X_temp,
    y_temp,

    test_size=0.50,

    random_state=42,

    stratify=y_temp
)


print(
    "Training samples  :",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)

print(
    "Test samples      :",
    len(X_test)
)


print(
    "\nTraining labels:"
)

print(
    y_train.value_counts()
)

Training samples  : 1127
Validation samples: 241
Test samples      : 242

Training labels:
label
1    707
0    420
Name: count, dtype: int64


In [30]:
# ============================================================
# CELL 13 — TRAIN RANDOM FOREST
# ============================================================

model = RandomForestClassifier(

    n_estimators=400,

    class_weight="balanced",

    min_samples_leaf=2,

    random_state=42,

    n_jobs=-1
)


model.fit(
    X_train,
    y_train
)


print(
    "================================"
)

print(
    "MODEL TRAINING COMPLETE"
)

print(
    "================================"
)

MODEL TRAINING COMPLETE


In [31]:
# ============================================================
# CELL 14 — DETERMINE RISK THRESHOLD
# ============================================================

classes = list(
    model.classes_
)


print(
    "Model classes:",
    classes
)


if 1 not in classes:

    raise ValueError(
        "Ransomware class (1) not found."
    )


ransomware_index = classes.index(
    1
)


val_prob = model.predict_proba(
    X_val
)[:, ransomware_index]


threshold_results = []


for threshold in np.arange(
    0.10,
    0.91,
    0.01
):

    predictions = (
        val_prob >= threshold
    ).astype(int)


    threshold_results.append({

        "threshold":
            threshold,

        "precision":
            precision_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_val,
                predictions,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_val,
                predictions,
                zero_division=0
            )
    })


threshold_df = pd.DataFrame(
    threshold_results
)


best = threshold_df.loc[
    threshold_df["f1"].idxmax()
]


BEST_THRESHOLD = float(
    best["threshold"]
)


print(
    "Best probability threshold:",
    round(
        BEST_THRESHOLD,
        4
    )
)


print(
    "Corresponding risk score:",
    round(
        BEST_THRESHOLD * 100,
        2
    )
)


print(
    "Validation F1:",
    round(
        best["f1"],
        4
    )
)

Model classes: [np.int64(0), np.int64(1)]
Best probability threshold: 0.23
Corresponding risk score: 23.0
Validation F1: 0.9868


In [32]:
# ============================================================
# CELL 15 — TEST SET EVALUATION
# ============================================================

test_prob = model.predict_proba(
    X_test
)[:, ransomware_index]


test_pred = (
    test_prob >= BEST_THRESHOLD
).astype(int)


accuracy = accuracy_score(
    y_test,
    test_pred
)


precision = precision_score(
    y_test,
    test_pred,
    zero_division=0
)


recall = recall_score(
    y_test,
    test_pred,
    zero_division=0
)


f1 = f1_score(
    y_test,
    test_pred,
    zero_division=0
)


print(
    "================================"
)

print(
    "FINAL MODEL RESULTS"
)

print(
    "================================"
)


print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        test_pred,
        target_names=[
            "Benign",
            "Ransomware"
        ],
        zero_division=0
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

FINAL MODEL RESULTS
Accuracy : 0.9876
Precision: 0.9869
Recall   : 0.9934
F1 Score : 0.9902

Classification Report:
              precision    recall  f1-score   support

      Benign       0.99      0.98      0.98        90
  Ransomware       0.99      0.99      0.99       152

    accuracy                           0.99       242
   macro avg       0.99      0.99      0.99       242
weighted avg       0.99      0.99      0.99       242


Confusion Matrix:
[[ 88   2]
 [  1 151]]


In [33]:
# ============================================================
# CELL 16 — FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    "feature":
        FINAL_FEATURES,

    "importance":
        model.feature_importances_

})


importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    importance_df
)# ============================================================
# CELL 16 — FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    "feature":
        FINAL_FEATURES,

    "importance":
        model.feature_importances_

})


importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    importance_df
)

,feature,importance
0,ata_write_entropy_max,0.309498
1,ata_write_entropy_mean,0.109105
2,ata_read_bytes,0.065237
3,ata_read_operations,0.062971
4,mem_write_entropy_max,0.058897
5,ata_read_velocity,0.053485
6,ata_write_entropy_std,0.047912
7,total_unique_addresses,0.029096
8,ata_write_bytes,0.027632
9,ata_write_operations,0.027457


,feature,importance
0,ata_write_entropy_max,0.309498
1,ata_write_entropy_mean,0.109105
2,ata_read_bytes,0.065237
3,ata_read_operations,0.062971
4,mem_write_entropy_max,0.058897
5,ata_read_velocity,0.053485
6,ata_write_entropy_std,0.047912
7,total_unique_addresses,0.029096
8,ata_write_bytes,0.027632
9,ata_write_operations,0.027457


In [34]:
# ============================================================
# CELL 17 — EXTRACT INDEPENDENT EXTRA DATASET
# ============================================================

extra_records = []

extra_samples = {
    sample_dir: traces
    for sample_dir, traces in sample_dirs.items()
    if get_dataset_group(sample_dir) == "extra"
}

print(
    "Extra samples found:",
    len(extra_samples)
)


for index, (
    sample_dir,
    traces
) in enumerate(
    extra_samples.items()
):

    if index % 25 == 0:
        print(
            f"Processing extra sample "
            f"{index}/{len(extra_samples)}"
        )


    # Extract EXACTLY the same features
    # used during training.

    features = build_sample_features(
        sample_dir,
        traces
    )


    # Get the already verified label

    label = get_label(
        sample_dir
    )


    features["label"] = label

    features["group"] = "extra"

    features["application"] = (
        get_application(sample_dir)
    )


    extra_records.append(
        features
    )


extra_df = pd.DataFrame(
    extra_records
)


print("\n================================")
print("EXTRA DATASET EXTRACTION COMPLETE")
print("================================")

print(
    "Samples:",
    len(extra_df)
)

print(
    "\nLabels:"
)

print(
    extra_df["label"].value_counts()
)

Extra samples found: 360
Processing extra sample 0/360
Processing extra sample 25/360
Processing extra sample 50/360
Processing extra sample 75/360
Processing extra sample 100/360
Processing extra sample 125/360
Processing extra sample 150/360
Processing extra sample 175/360
Processing extra sample 200/360
Processing extra sample 225/360
Processing extra sample 250/360
Processing extra sample 275/360
Processing extra sample 300/360
Processing extra sample 325/360
Processing extra sample 350/360

EXTRA DATASET EXTRACTION COMPLETE
Samples: 360

Labels:
label
1    210
0    150
Name: count, dtype: int64


In [35]:
# ============================================================
# CELL 18 — PREPARE EXTRA FEATURES
# ============================================================

extra_df = extra_df.replace(
    [np.inf, -np.inf],
    np.nan
)

extra_df = extra_df.fillna(0)


X_extra = extra_df[
    FINAL_FEATURES
]

y_extra = extra_df[
    "label"
].astype(int)


print(
    "Extra feature shape:",
    X_extra.shape
)

print(
    "Expected:",
    (360, len(FINAL_FEATURES))
)

Extra feature shape: (360, 27)
Expected: (360, 27)


In [36]:
# ============================================================
# CELL 19 — INDEPENDENT EXTRA EVALUATION
# ============================================================

extra_prob = model.predict_proba(
    X_extra
)[:, ransomware_index]


# Use the SAME threshold selected earlier

extra_pred = (
    extra_prob >= BEST_THRESHOLD
).astype(int)


extra_accuracy = accuracy_score(
    y_extra,
    extra_pred
)

extra_precision = precision_score(
    y_extra,
    extra_pred,
    zero_division=0
)

extra_recall = recall_score(
    y_extra,
    extra_pred,
    zero_division=0
)

extra_f1 = f1_score(
    y_extra,
    extra_pred,
    zero_division=0
)


print(
    "================================"
)

print(
    "INDEPENDENT EXTRA EVALUATION"
)

print(
    "================================"
)

print(
    f"Accuracy : {extra_accuracy:.4f}"
)

print(
    f"Precision: {extra_precision:.4f}"
)

print(
    f"Recall   : {extra_recall:.4f}"
)

print(
    f"F1 Score : {extra_f1:.4f}"
)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_extra,
        extra_pred,
        target_names=[
            "Benign",
            "Ransomware"
        ],
        zero_division=0
    )
)


print(
    "\nConfusion Matrix:"
)

print(
    confusion_matrix(
        y_extra,
        extra_pred
    )
)

INDEPENDENT EXTRA EVALUATION
Accuracy : 0.9361
Precision: 0.9013
Recall   : 1.0000
F1 Score : 0.9481

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.85      0.92       150
  Ransomware       0.90      1.00      0.95       210

    accuracy                           0.94       360
   macro avg       0.95      0.92      0.93       360
weighted avg       0.94      0.94      0.94       360


Confusion Matrix:
[[127  23]
 [  0 210]]


In [37]:
# ============================================================
# CELL 17 — SAVE FINAL MODEL
# ============================================================

MODEL_PATH = (
    "/kaggle/working/"
    "filesystem_model.pkl"
)


METADATA_PATH = (
    "/kaggle/working/"
    "filesystem_model_metadata.json"
)


# ------------------------------------------------------------
# Save model
# ------------------------------------------------------------

joblib.dump(
    model,
    MODEL_PATH
)


# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

metadata = {

    "model_name":
        "TRINETRA Risk Analyser",

    "model_version":
        "v1",

    "model_type":
        "RandomForestClassifier",

    "features":
        FINAL_FEATURES,

    "feature_count":
        len(FINAL_FEATURES),

    "risk_score":
        "ransomware_probability * 100",

    "operational_threshold":
        44,

    "threshold_rule":
        "risk_score >= 44",

    "validation_probability_threshold":
        BEST_THRESHOLD,

    "source_dataset":
        "RanSMAP 2024",

    "training_groups":
        [
            "original",
            "variants",
            "mix"
        ],

    "excluded_from_training":
        [
            "extra"
        ],

    "training_samples":
        len(behavior_df),

    "training_benign":
        int(
            (y == 0).sum()
        ),

    "training_ransomware":
        int(
            (y == 1).sum()
        ),

    "telemetry_sources":
        [
            "ATA read",
            "ATA write",
            "memory read",
            "memory write",
            "memory read/write",
            "memory execute"
        ],

    "note":
        (
            "Prototype ransomware behavioral model "
            "trained on aggregated RanSMAP telemetry. "
            "Runtime RiskAnalyser must receive the "
            "same feature schema for valid inference."
        )
}


with open(
    METADATA_PATH,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


print(
    "Model saved:"
)

print(
    MODEL_PATH
)


print(
    "\nMetadata saved:"
)

print(
    METADATA_PATH
)

Model saved:
/kaggle/working/filesystem_model.pkl

Metadata saved:
/kaggle/working/filesystem_model_metadata.json


In [38]:
# ============================================================
# CELL 18 — DOWNLOAD OUTPUTS
# ============================================================

from IPython.display import FileLink


print(
    "DOWNLOAD FILES"
)

print(
    "=============="
)


display(
    FileLink(
        MODEL_PATH
    )
)


display(
    FileLink(
        METADATA_PATH
    )
)


display(
    FileLink(
        TRAINING_DATA_PATH
    )
)

DOWNLOAD FILES


/kaggle/working/filesystem_model.pkl

/kaggle/working/filesystem_model_metadata.json

/kaggle/working/ransmap_behavior_features.csv